In [ ]:
import os, json, boto3
from opensearchpy import OpenSearch
from dotenv import load_dotenv
load_dotenv()

INDEX_NAME    = "listings"
PIPELINE_NAME = "hybrid-rrf-pipeline"
EMBED_MODEL   = "amazon.titan-embed-text-v2:0"
EMBED_DIM     = 1024
REGION        = os.environ.get("AWS_DEFAULT_REGION", "eu-central-2")

In [ ]:
host = os.environ["OPENSEARCH_ENDPOINT"]
auth = (os.environ["OPENSEARCH_USER"], os.environ["OPENSEARCH_PW"])

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth,
    http_compress=True,
    use_ssl=True,
    verify_certs=True,
    ssl_assert_hostname=False,
    ssl_show_warn=False,
)

In [ ]:
def embed(text: str) -> list[float]:
    bedrock = boto3.client("bedrock-runtime", region_name=REGION)
    resp = bedrock.invoke_model(
        modelId=EMBED_MODEL,
        body=json.dumps({"inputText": text, "dimensions": EMBED_DIM, "normalize": True}),
        contentType="application/json",
        accept="application/json",
    )
    return json.loads(resp["body"].read())["embedding"]

In [ ]:
query_text = "2 bedroom apartment in Zurich under 2500 CHF"
vector = embed(query_text)

resp = client.search(
    index=INDEX_NAME,
    body={
        "size": 10,
        "query": {
            "hybrid": {
                "queries": [
                    {"multi_match": {"query": query_text, "fields": ["full_text", "title", "description"]}},
                    {"knn": {"dense_embedding": {"vector": vector, "k": 10}}},
                ]
            }
        },
    },
    params={"search_pipeline": PIPELINE_NAME},
)

for hit in resp["hits"]["hits"]:
    s = hit["_source"]
    print(f"[{hit['_score']:.4f}] {s['title']} — {s['city']} — CHF {s['price']}")